# HCAD Fixture Features, 2021–2025
This notebook extends the 2021 HCAD fixture-processing methodology to 2022–2025. fixtures.txt contains building-level characteristics such as bedrooms, bathrooms, rooms, stories, parking, apartment-unit composition, and other physical features. Because one HCAD account can contain multiple buildings and the same fixture type can appear more than once within a building, the raw fixture file cannot be merged directly into the property-level dataset.
The data are first reduced to one value for each acct × bld_num × type by averaging repeated records of the same fixture type within a building. Building-level values are then averaged across buildings belonging to the same HCAD account, producing one property-level fixture profile per account. The same procedure is applied to all five years.
The core fixture variables follow the 2021 analysis. AP0–AP3 and net rentable area are also retained to better represent apartment properties. AP0–AP3 are additionally used to calculate the average number of bedrooms per apartment unit within each building before buildings are averaged to the property level.
Missing fixture types are not inferred. A numeric value in units cannot be interpreted without knowing its fixture type, so records missing the fixture type are excluded and counted as data loss. Missing property features remain missing rather than being replaced with zero or with averages from other properties.
## Handling Apartment data
While reviewing and analyzing the 2021 HCAD data, some problem arose particularly with the apartment data. When handling property features- number of bed, baths, parking, and other special feature - we need to recognize the dimensionality of apartment information. Most apartment data is sectioned by building, which can later become dificult to connect with JPL eviction data, but it can also be difficult to handle by the GNN model. We created a mean profile for the building and used that as the proxy for the entire building units. 

**State Class**

A1 = Residential, Single-Family\
B1 = Residential, Multi-Family\
B2 = Residential, Two-Family\
B3 = Residential, Three-Family\
B4 = Residential, Four- or More-Family

**B1-B4**

1002 = Duplex\
1003 = Triplex\
1004 = Fourplex

4209 = Apartment Structure, 4–20 units\
4211 = Apartment Garden\
4212 = Apartment Mid Rise\
4213 = Manufactured Housing Park\
4214 = Apartment High Rise\
4221 = Subsidized Housing\
4222 = Apartment - Tax Credit\
4299 = Apartment Structure

**PGI** 

13 = Manufactured Housing Parks\
16 = Apartments - Mid Rise to High Rise\
17 = Apartments - Garden Style\
18 = Apartments - Tax Credit Housing\
19 = Apartments - Subsidized Housing

In [80]:
from pathlib import Path
import pandas as pd
import numpy as np
import gc

In [81]:
project = Path(r"C:\Users\alys_\Documents\Research\Evictions")
hcad_root = project / "01_DATA_RAW" / "HCAD"
output_root = project / "02_DATA_PROCESSED" / "fixtures"

output_root.mkdir(parents=True, exist_ok=True)

In [82]:
years = [2021, 2022, 2023, 2024, 2025]

codes = [
    "RMB", "RMF", "RMH", "RMR", "RMT",
    "STC", "PRK", "UNT",
    "AC1", "AC2",
    "FL1", "FL2", "FL3",
    "FPD", "FPM", "FPO", "FPW",
    "AP0", "AP1", "AP2", "AP3",
    "NRA"
]

ap_codes = ["AP0", "AP1", "AP2", "AP3"]

In [83]:
def load_fixtures(year):
    path = hcad_root / f"pdata{year}" / "fixtures.txt"

    df = pd.read_csv(
        path,
        sep="\t",
        dtype=str,
        encoding="latin1",
        usecols=["acct", "bld_num", "type", "type_dscr", "units"],
        low_memory=False
    )

    for col in ["acct", "bld_num", "type", "type_dscr"]:
        df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

    df["units"] = pd.to_numeric(df["units"], errors="coerce")

    return df

In [84]:
def make_fixtures(year):
    df = load_fixtures(year)

    raw_rows = len(df)
    raw_accounts = df["acct"].nunique()

    missing_acct = df["acct"].isna().sum()
    missing_bld = df["bld_num"].isna().sum()
    missing_type = df["type"].isna().sum()
    missing_units = df["units"].isna().sum()

    good = (
        df["acct"].notna()
        & df["bld_num"].notna()
        & df["type"].notna()
        & df["units"].notna()
    )

    excluded = df[~good].copy()
    df = df[good].copy()

    grouped = (
        df.groupby(["acct", "bld_num", "type"], as_index=False)
        .agg(
            units=("units", "mean"),
            n=("units", "size")
        )
    )

    repeated_groups = (grouped["n"] > 1).sum()

    base = (
        df.groupby("acct")["bld_num"]
        .nunique()
        .rename("n_buildings")
        .reset_index()
    )

    selected = grouped[grouped["type"].isin(codes)].copy()

    buildings = (
        selected.pivot(
            index=["acct", "bld_num"],
            columns="type",
            values="units"
        )
        .reset_index()
    )

    buildings.columns.name = None

    for code in codes:
        if code not in buildings.columns:
            buildings[code] = np.nan

    ap = buildings[ap_codes]
    has_ap = ap.notna().any(axis=1)
    ap = ap.fillna(0)

    apt_units = ap.sum(axis=1).where(has_ap)

    apt_bedrooms = (
        ap["AP1"]
        + 2 * ap["AP2"]
        + 3 * ap["AP3"]
    )

    buildings["apt_avg_bedrooms"] = (
        apt_bedrooms / apt_units
    ).where(apt_units > 0)

    features = codes + ["apt_avg_bedrooms"]

    account = (
        buildings.groupby("acct")[features]
        .mean()
        .reset_index()
    )

    out = base.merge(
        account,
        on="acct",
        how="left",
        validate="one_to_one"
    )

    out.columns = [str(col).lower() for col in out.columns]

    feature_cols = [col.lower() for col in features]

    selected_accounts = (
        out[feature_cols]
        .notna()
        .any(axis=1)
        .sum()
    )

    lost_accounts = raw_accounts - len(out)

    out.to_parquet(
        output_root / f"fixtures_{year}.parquet",
        index=False
    )

    if len(excluded) > 0:
        excluded.to_csv(
            output_root / f"fixtures_excluded_{year}.csv",
            index=False
        )

    missing = (
        out[feature_cols]
        .isna()
        .mean()
        .mul(100)
        .round(2)
        .rename("missing_pct")
        .reset_index()
        .rename(columns={"index": "feature"})
    )

    missing["year"] = year

    summary = {
        "year": year,
        "raw_rows": raw_rows,
        "raw_accounts": raw_accounts,
        "missing_acct": missing_acct,
        "missing_bld": missing_bld,
        "missing_type": missing_type,
        "missing_units": missing_units,
        "excluded_rows": len(excluded),
        "repeated_groups": repeated_groups,
        "final_accounts": len(out),
        "selected_accounts": selected_accounts,
        "lost_accounts": lost_accounts
    }

    return summary, missing

In [85]:
summary = []
missing = []

for year in years:
    print("processing", year)

    year_summary, year_missing = make_fixtures(year)

    summary.append(year_summary)
    missing.append(year_missing)

    gc.collect()

processing 2021
processing 2022
processing 2023
processing 2024
processing 2025


In [86]:
df_summary = pd.DataFrame(summary)

df_missing = pd.concat(
    missing,
    ignore_index=True
)

df_summary.to_csv(
    output_root / "fixtures_summary.csv",
    index=False
)

df_missing.to_csv(
    output_root / "fixtures_missingness.csv",
    index=False
)

In [87]:
display(df_summary)

,year,raw_rows,raw_accounts,missing_acct,missing_bld,missing_type,missing_units,excluded_rows,repeated_groups,final_accounts,selected_accounts,lost_accounts
0,2021,7573519,1267694,0,0,12,0,12,29426,1267694,1210964,0
1,2022,7694168,1286699,0,0,12,0,12,29955,1286699,1229514,0
2,2023,7800912,1303855,0,0,10,0,10,30112,1303855,1246042,0
3,2024,7919738,1322238,0,0,9,0,9,30686,1322238,1264006,0
4,2025,8040796,1341218,0,0,8,0,8,31180,1341218,1282627,0


In [88]:
display(
    df_missing.pivot(
        index="feature",
        columns="year",
        values="missing_pct"
    )
)

year,2021,2022,2023,2024,2025
feature,,,,,
ac1,100.00,100.00,100.00,100.00,100.00
ac2,100.00,100.00,100.00,100.00,100.00
ap0,99.96,99.96,99.96,99.96,99.96
ap1,99.76,99.76,99.76,99.76,99.77
ap2,99.77,99.77,99.77,99.77,99.77
ap3,99.91,99.91,99.91,99.91,99.91
apt_avg_bedrooms,99.81,99.81,99.81,99.81,99.81
fl1,99.99,99.99,99.99,99.99,99.99
fl2,100.00,100.00,100.00,100.00,100.00


In [89]:
df_fixtures_2022 = pd.read_parquet(
    output_root / "fixtures_2022.parquet"
)

display(df_fixtures_2022.head())

display(
    df_fixtures_2022.describe().T
)

,acct,n_buildings,rmb,rmf,rmh,rmr,rmt,stc,prk,unt,...,fpd,fpm,fpo,fpw,ap0,ap1,ap2,ap3,nra,apt_avg_bedrooms
0,0010020000016,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0010040000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0010100000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0010120000010,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0010150000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
n_buildings,1286699.0,1.080499,1.188546,1.0,1.000000,1.000000,1.000000,313.000
rmb,1219188.0,3.239955,0.828572,0.0,3.000000,3.000000,4.000000,20.000
rmf,1219136.0,2.037311,0.716584,0.0,2.000000,2.000000,2.000000,11.000
rmh,506950.0,1.035809,0.211181,0.0,1.000000,1.000000,1.000000,7.000
rmr,447169.0,1.242285,0.568756,0.0,1.000000,1.000000,1.000000,13.000
rmt,1219053.0,6.630652,1.644772,0.0,6.000000,6.000000,8.000000,40.000
stc,9877.0,2.844691,3.646924,0.0,2.000000,2.000000,3.000000,75.000
prk,415.0,403.615422,696.157220,0.0,29.500000,267.000000,506.000000,10623.000
unt,624.0,165.105057,152.931854,0.0,13.750000,135.500000,294.000000,1248.000
ac1,28.0,2425.120000,5589.322515,0.0,1.000000,1.000000,975.000000,22000.000


In [90]:
df_fixtures_2023 = pd.read_parquet(
    output_root / "fixtures_2023.parquet"
)

df_fixtures_2024 = pd.read_parquet(
    output_root / "fixtures_2024.parquet"
)

df_fixtures_2025 = pd.read_parquet(
    output_root / "fixtures_2025.parquet"
)

In [91]:
display(df_fixtures_2023.head())

display(
    df_fixtures_2023.describe().T
)

,acct,n_buildings,rmb,rmf,rmh,rmr,rmt,stc,prk,unt,...,fpd,fpm,fpo,fpw,ap0,ap1,ap2,ap3,nra,apt_avg_bedrooms
0,0010020000016,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0010040000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0010100000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0010120000010,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0010150000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
n_buildings,1303855.0,1.080083,1.188502,1.0,1.000000,1.000000,1.000000,313.0
rmb,1235606.0,3.247351,0.829356,0.0,3.000000,3.000000,4.000000,20.0
rmf,1235555.0,2.047787,0.720835,0.0,2.000000,2.000000,2.000000,11.0
rmh,516929.0,1.036300,0.212505,0.0,1.000000,1.000000,1.000000,7.0
rmr,454250.0,1.247819,0.575371,0.0,1.000000,1.000000,1.000000,13.0
rmt,1235474.0,6.640051,1.643755,0.0,6.000000,6.000000,8.000000,40.0
stc,9980.0,2.861471,3.660827,0.0,2.000000,2.000000,3.000000,75.0
prk,428.0,408.185514,690.381557,0.0,30.000000,277.500000,509.000000,10623.0
unt,680.0,172.391786,156.172600,0.0,16.000000,146.000000,300.000000,1248.0
ac1,27.0,2514.925926,5675.171932,0.0,1.000000,1.000000,1850.000000,22000.0


In [93]:
dfs = {
    2022: df_fixtures_2022,
    2023: df_fixtures_2023,
    2024: df_fixtures_2024,
    2025: df_fixtures_2025
}

rows = []

for year, df in dfs.items():
    for col in ["ap0", "ap1", "ap2", "ap3", "unt"]:
        rows.append({
            "year": year,
            "feature": col,
            "count": df[col].count(),
            "max": df[col].max(),
            'min':df[col].min(),
            "p99": df[col].quantile(.99)
        })

df_ap_check = pd.DataFrame(rows)

display(df_ap_check)

bad_accts = (
    df_fixtures_2023
    .sort_values("ap2", ascending=False)
    .head(10)["acct"]
    .tolist()
)

display(
    df_fixtures_2023[
        df_fixtures_2023["acct"].isin(bad_accts)
    ][
        [
            "acct",
            "n_buildings",
            "ap0",
            "ap1",
            "ap2",
            "ap3",
            "unt",
            "apt_avg_bedrooms"
        ]
    ]
)

df_raw_2023 = load_fixtures(2023)

display(
    df_raw_2023[
        df_raw_2023["acct"].isin(bad_accts)
        & df_raw_2023["type"].isin(
            ["AP0", "AP1", "AP2", "AP3", "UNT"]
        )
    ].sort_values(
        ["acct", "bld_num", "type"]
    )
)

,year,feature,count,max,min,p99
0,2022,ap0,489,300.0,0.0,168.00
1,2022,ap1,3061,1450.0,0.0,296.20
2,2022,ap2,2962,434.0,0.0,192.00
3,2022,ap3,1121,355.0,0.0,110.40
4,2022,unt,624,1248.0,0.0,501.72
5,2023,ap0,493,300.0,0.0,168.16
6,2023,ap1,3086,145419.0,0.0,298.30
7,2023,ap2,2988,245623.0,0.0,194.26
8,2023,ap3,1136,29043.0,0.0,111.65
9,2023,unt,680,1248.0,0.0,550.50


,acct,n_buildings,ap0,ap1,ap2,ap3,unt,apt_avg_bedrooms
69315,0410210010279,13,NaN,4.0,243.0,61.0,NaN,2.185065
75960,0422240000205,38,52.0,52.0,240.0,48.0,NaN,1.724490
80396,0432110000039,42,NaN,8.0,434.0,12.0,NaN,2.008811
89315,0461710000011,54,NaN,204.0,292.0,20.0,NaN,1.643411
845024,1182710000001,29,42.0,276.0,324.0,36.0,NaN,1.522124
1003908,1255090010001,1,NaN,126.0,263.0,42.0,NaN,1.805104
1088504,1290860010001,2,NaN,175.0,248.0,NaN,NaN,1.586288
1125393,1316370010001,12,NaN,60.0,274.0,32.0,NaN,1.923497
1258587,1448650010003,12,NaN,60.0,264.0,NaN,324.0,1.814815
1264602,1455620010001,1,NaN,145419.0,245623.0,29043.0,317.0,1.722970


,acct,bld_num,type,type_dscr,units
396761,0410210010279,1,AP1,Apt: 1-Bedroom Unit,4.0
396759,0410210010279,1,AP2,Apt: 2-Bedroom Unit,243.0
396762,0410210010279,1,AP3,Apt: 3-Bedroom Unit,61.0
446294,0422240000205,1,AP0,Apt: Effcncy Unit,52.0
446295,0422240000205,1,AP1,Apt: 1-Bedroom Unit,52.0
446296,0422240000205,1,AP2,Apt: 2-Bedroom Unit,240.0
446297,0422240000205,1,AP3,Apt: 3-Bedroom Unit,48.0
473159,0432110000039,1,AP1,Apt: 1-Bedroom Unit,8.0
473160,0432110000039,1,AP2,Apt: 2-Bedroom Unit,434.0
473161,0432110000039,1,AP3,Apt: 3-Bedroom Unit,12.0


In [94]:
acct = "1455620010001"

for year in [2022, 2023, 2024, 2025]:
    df = load_fixtures(year)

    check = df[
        (df["acct"] == acct) &
        (df["type"].isin(["AP0", "AP1", "AP2", "AP3", "UNT"]))
    ][
        ["acct", "bld_num", "type", "type_dscr", "units"]
    ]

    print("\n", year)
    display(check)


 2022


,acct,bld_num,type,type_dscr,units



 2023


,acct,bld_num,type,type_dscr,units
7627122,1455620010001,1,UNT,Number of Apartment Units,317.0
7627124,1455620010001,1,AP1,Apt: 1-Bedroom Unit,145419.0
7627125,1455620010001,1,AP2,Apt: 2-Bedroom Unit,245623.0
7627126,1455620010001,1,AP3,Apt: 3-Bedroom Unit,29043.0



 2024


,acct,bld_num,type,type_dscr,units
7681211,1455620010001,1,UNT,Number of Apartment Units,317.0
7681213,1455620010001,1,AP1,Apt: 1-Bedroom Unit,145419.0
7681214,1455620010001,1,AP2,Apt: 2-Bedroom Unit,245623.0
7681215,1455620010001,1,AP3,Apt: 3-Bedroom Unit,29043.0



 2025


,acct,bld_num,type,type_dscr,units
7734456,1455620010001,1,UNT,Number of Apartment Units,317.0
7734458,1455620010001,1,AP1,Apt: 1-Bedroom Unit,145419.0
7734459,1455620010001,1,AP2,Apt: 2-Bedroom Unit,245623.0
7734460,1455620010001,1,AP3,Apt: 3-Bedroom Unit,29043.0


In [95]:
bad_acct = "1455620010001"

for df in [
    df_fixtures_2023,
    df_fixtures_2024,
    df_fixtures_2025
]:
    df.loc[
        df["acct"] == bad_acct,
        ["ap0", "ap1", "ap2", "ap3", "apt_avg_bedrooms"]
    ] = np.nan

In [96]:
df_fixtures_2023.to_parquet(
    output_root / "fixtures_2023.parquet",
    index=False
)

df_fixtures_2024.to_parquet(
    output_root / "fixtures_2024.parquet",
    index=False
)

df_fixtures_2025.to_parquet(
    output_root / "fixtures_2025.parquet",
    index=False
)

## Results and decisions
The 2021 fixture-processing methodology was extended consistently through 2025. Raw fixture records increased from 7,573,519 in 2021 to 8,040,796 in 2025, while accounts represented increased from 1,267,694 to 1,341,218. Only 8–12 records per year lacked a fixture type and were excluded because their numeric values could not be interpreted. No HCAD accounts were lost entirely in any year.
Approximately 29,000–31,000 repeated acct × bld_num × type groups occurred each year. Repeated values of the same fixture type within a building were averaged first, and building-level profiles were then averaged to one account-level profile.
Residential variables such as bedrooms, full bathrooms, and total rooms had high coverage, while apartment-specific and commercial fixture variables were sparse in the full HCAD population. This sparsity is expected because these variables apply only to particular property types and will be reassessed within the rental and apartment cohorts.
One account, 1455620010001, contained implausible AP1–AP3 values beginning in 2023 despite a reported apartment-unit count of 317. The anomalous values were present in the raw HCAD source files for 2023–2025, confirming that they were not created by the aggregation procedure. The affected apartment-composition fields were therefore set to missing while the property itself was retained.